In [3]:
import pandas as pd
import numpy as np

In [2]:
class Option:
    def __init__(self,
                 strike: float,
                 spot: float,
                 option_type: str,
                 premium: float,
                 expiry: str):
        self.strike = strike
        self.spot_price = spot
        self.option_type = option_type
        self.premium = premium
        self.expiry = expiry
    
    def short_call(self):
        return np.minimum(self.premium, self.premium-np.maximum(0, self.spot_price - self.strike))
    
    def long_call(self):
        return np.maximum(self.spot_price - self.strike - self.premium)
    
    def short_put(self):
        return np.minimum(self.premium, self.premium-np.maximum(0, self.strike - self.spot_price))
    
    def long_put(self):
        return np.maximum(self.strike - self.spot_price - self.premium, -self.premium)

class OptionsBacktester:
    def __init__(self,
                 signal_df: pd.DataFrame,
                 spot_price_df: pd.DataFrame,
                 options_df: pd.DataFrame,
                 upper_threshold: float,
                 lower_threshold: float,
                 option_type: str,
                 dynamic_option_strategy: bool = False,
                 dynamic_strike: bool = False,):
        self.signal_df = signal_df
        self.signal_df.columns = ["signal"]
        self.spot_price_df = spot_price_df
        self.options_df = options_df
        self.upper_threshold = upper_threshold
        self.lower_threshold = lower_threshold
        self.option_type = option_type
        self.dynamic_option_strategy = dynamic_option_strategy
        self.dynamic_strike = dynamic_strike

        self.comb_df = pd.concat([self.signal_df, self.spot_price_df.shift(), self.options_df.shift()], axis=1)
        self.comb_df["action"] = np.where(self.comb_df["signal"] > self.upper_threshold, 1, np.where(self.comb_df["signal"] < self.lower_threshold, -1, 0))

    def long_short_vol_df(self):
        self.trading_df = {}
        for index, row in self.comb_df.iterrows():
            if row["action"] != 0:
                self.trading_df[index] = self.comb_df.loc[index]

    def iron_condor(self, direction: str):
        """
        Long (Short)
        1. Short (Long) 1 OTM Put
        2. Long (Short) 1 close to ATM Put
        3. Long (Short) 1 close to ATM Call
        4. Short (Long) 1 OTM Call
        """
        if direction == "long":
            short_put = Option(self.comb_df["strike"], self.comb_df["spot_price"], self.option_type, self.comb_df["premium"], self.comb_df["expiry"]).short_call() # need to edit the column names
            long_put = Option(self.comb_df["strike"], self.comb_df["spot_price"], self.option_type, self.comb_df["premium"], self.comb_df["expiry"]).long_put()
            long_call = Option(self.comb_df["strike"], self.comb_df["spot_price"], self.option_type, self.comb_df["premium"], self.comb_df["expiry"]).long_call()
            short_call = Option(self.comb_df["strike"], self.comb_df["spot_price"], self.option_type, self.comb_df["premium"], self.comb_df["expiry"]).short_call()
            payoff = short_put + long_put + long_call + short_call
        elif direction == "short":
            long_put = Option(self.comb_df["strike"], self.comb_df["spot_price"], self.option_type, self.comb_df["premium"], self.comb_df["expiry"]).long_put()
            short_put = Option(self.comb_df["strike"], self.comb_df["spot_price"], self.option_type, self.comb_df["premium"], self.comb_df["expiry"]).short_put()
            short_call = Option(self.comb_df["strike"], self.comb_df["spot_price"], self.option_type, self.comb_df["premium"], self.comb_df["expiry"]).short_call()
            long_call = Option(self.comb_df["strike"], self.comb_df["spot_price"], self.option_type, self.comb_df["premium"], self.comb_df["expiry"]).long_call()
            payoff = long_put + short_put + short_call + long_call

SyntaxError: non-default argument follows default argument (1068281392.py, line 9)